In [ ]:
import os
import json
import mlflow
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
load_dotenv('/home/tinhanhnguyen/Desktop/HK8/Capstone/CAPSTONE_PROJECT/videodeepsearch/.env')
from mlflow.genai.scorers import (
    ToolCallEfficiency,
    ToolCallCorrectness,
    Correctness,
    RelevanceToQuery,
)
from mlflow.genai.scorers.deepeval import (
    TaskCompletion,
)

mlflow.set_tracking_uri("http://100.113.186.28:5000")
mlflow.set_experiment("vds-agent-validation")

<Experiment: artifact_location='mlflow-artifacts:/6', creation_time=1775830300571, experiment_id='6', last_update_time=1775830300571, lifecycle_stage='active', name='vds-agent-validation', tags={}, workspace='default'>

In [28]:
import pandas as pd

from mlflow.entities import Trace
from typing import cast

def get_record_based_on_trace(
    data_df: list[dict],
    trace: Trace
): 
    """
    Get the corresponding record based on trace's session id
    """
    trace_input_preview = cast(str, trace.info.request_preview).strip('"').strip("'").replace('\\n', '')
    
    filter_data_record = next(
        filter(
            lambda x: x['inputs']['user_demand'].replace('\n', '') == trace_input_preview, data_df
        )
    )
    
    return filter_data_record

In [30]:
key = os.getenv("OPENROUTER_API_KEY")
os.environ["OPENROUTER_API_KEY"] = key
os.environ["OPENAI_API_KEY"] = key

In [ ]:
judge_model = "openrouter:/qwen/qwen3.6-plus"
scorers = [
    TaskCompletion(model=judge_model, threshold=0.5, include_reason=True),
    ToolCallEfficiency(model=judge_model),
    ToolCallCorrectness(model=judge_model),
    RelevanceToQuery(model=judge_model),
    Correctness(model=judge_model),
]

In [5]:
import json
dataset = mlflow.genai.datasets.get_dataset(dataset_id="d-ec37df2ccdfa4ce5b9614724fdceb27e")
dataset_df = json.loads(dataset.to_json())
type(dataset_df)

dict

In [6]:
len(dataset_df['records'])

317

In [ ]:
traces = mlflow.search_traces(
    locations=['6'],
    filter_string="trace.text LIKE '%I want%'"
)

In [8]:
len(traces)

70

In [9]:
trace_id = traces.iloc[-1]["trace_id"]
sample_trace = mlflow.get_trace(trace_id)

In [27]:
sample_trace.info.request_preview.strip('"').strip("'").replace('\\n', '') == dataset_df['records'][0]['inputs']['user_demand'].replace('\n', '')

True

In [63]:
sample_trace.info

TraceInfo(trace_id='tr-8fafb04c98bd4e967d79ff101ace7c43', trace_location=TraceLocation(type=<TraceLocationType.MLFLOW_EXPERIMENT: 'MLFLOW_EXPERIMENT'>, mlflow_experiment=MlflowExperimentLocation(experiment_id='6'), inference_table=None, uc_schema=None), request_time=1775905661360, state=<TraceState.OK: 'OK'>, request_preview='"\\n        I want to find the moment related to this topics: Cooking Tutorials. Try all your possible best, and answer this question:\\n        What are the essential knife skills for different vegetables to ensure uniform cuts and safety?\\n\\n        The answer must be including your final answers, related video ids and segment in (start_time, end_time) manner. \\n        "', response_preview='"I\'ve searched extensively across cooking tutorial videos for essential knife skills demonstrations. Here\'s what I found:\\n\\n---\\n\\n## 🔪 Essential Knife Skills for Different Vegetables: Complete Answer\\n\\n### 1. Proper Knife Grip & Handling\\n- **Three-finger grip

In [ ]:
get_record_based_on_trace(
    data_df=dataset_df['records'],
    trace=sample_trace
)

In [ ]:
tc_scorer = TaskCompletion(model=judge_model, threshold=0.5, include_reason=True) #type:ignore
tc_result = tc_scorer(trace=sample_trace)

In [32]:
tc_result

Feedback(name='TaskCompletion', source=AssessmentSource(source_type='LLM_JUDGE', source_id='openrouter:/qwen/qwen3.6-plus'), trace_id=None, run_id=None, rationale='The actual outcome perfectly aligns with the desired task. It successfully provides a comprehensive list of video segments complete with video IDs and timestamps, explicitly covers essential knife skills, safety guidelines, and specific techniques for achieving uniform cuts across a wide variety of vegetables, fulfilling every requirement specified in the task.', metadata={'score': 1.0, 'threshold': 0.5, 'mlflow.scorer.framework': 'deepeval'}, span_id=None, create_time_ms=1776000995332, last_update_time_ms=1776000995332, assessment_id=None, error=None, expectation=None, feedback=FeedbackValue(value=<CategoricalRating.YES: 'yes'>, error=None), overrides=None, valid=True)

In [70]:
tc_result

Feedback(name='TaskCompletion', source=AssessmentSource(source_type='LLM_JUDGE', source_id='openrouter:/qwen/qwen3.6-plus'), trace_id=None, run_id=None, rationale='The actual outcome perfectly aligns with the desired task. It successfully provides a comprehensive list of video segments complete with video IDs and timestamps, explicitly covers essential knife skills, safety guidelines, and specific techniques for achieving uniform cuts across a wide variety of vegetables, fulfilling every requirement specified in the task.', metadata={'score': 1.0, 'threshold': 0.5, 'mlflow.scorer.framework': 'deepeval'}, span_id=None, create_time_ms=1776000995332, last_update_time_ms=1776000995332, assessment_id=None, error=None, expectation=None, feedback=FeedbackValue(value=<CategoricalRating.YES: 'yes'>, error=None), overrides=None, valid=True)

In [47]:
tool_eff_scorer = ToolCallCorrectness(model=judge_model) #type:ignore
tool_eff_result = tool_eff_scorer(trace=sample_trace)

2026/04/12 20:45:55 WARNING mlflow.genai.utils.trace_utils: Failed to extract tools from trace using LLM. Returning empty list. Error: MlflowException('Failed to invoke the judge via litellm: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************25ae. You can find your API key at https://platform.openai.com/account/api-keys.')


In [48]:
tool_eff_result

Feedback(name='tool_call_correctness', source=AssessmentSource(source_type='LLM_JUDGE', source_id='openrouter:/qwen/qwen3.6-plus'), trace_id=None, run_id=None, rationale="The user's request is highly specific, asking for video segments on essential knife skills for vegetables, focusing on uniform cuts and safety, with exact start and end times. The agent correctly identifies the need for a comprehensive, multi-modal search. It appropriately uses a multi-worker architecture to divide the task into keyword/transcript search, visual/KG search, and a deep dive into safety/grip. The tools selected (`search_bm25`, `get_audio_from_query_hybrid`, `get_segments_from_event_query_mmbert`, `kg.search_entities_semantic`, `search.get_images_from_qwenvl_query`, `utility.get_related_asr_from_segment`, etc.) perfectly match the available capabilities for video retrieval. The arguments passed to these tools directly align with the user's request, using precise keywords like 'knife skills', 'julienne', '

In [51]:
tool_efficiency_scorer = ToolCallEfficiency(model=judge_model) #type:ignore
tool_efficiency_result = tool_efficiency_scorer(trace=sample_trace)

2026/04/12 20:48:02 WARNING mlflow.genai.utils.trace_utils: Failed to extract tools from trace using LLM. Returning empty list. Error: MlflowException('Failed to invoke the judge via litellm: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************25ae. You can find your API key at https://platform.openai.com/account/api-keys.')


In [52]:
tool_efficiency_result

Feedback(name='tool_call_efficiency', source=AssessmentSource(source_type='LLM_JUDGE', source_id='openrouter:/qwen/qwen3.6-plus'), trace_id=None, run_id=None, rationale='The agent initially spawns a search worker (Tool Call 5) that executes a comprehensive multi-tool search strategy and returns a detailed report of findings across BM25, audio hybrid, audio dense, and event searches. However, immediately after receiving these results, the agent manually calls numerous individual tools (Tool Calls 6-24) using nearly identical queries. Furthermore, the agent spawns two additional workers (Tool Calls 25 and 45) that perform yet another round of overlapping KG, visual, and audio searches for the exact same topic. This creates multiple redundant passes over the search space with highly similar parameters, failing to consolidate searches or utilize the initial results efficiently before triggering new, overlapping workflows.', metadata={'mlflow.assessment.judgeInputTokens': 249984, 'mlflow.as

In [15]:
sample_request_preview = sample_trace.info.request_preview
sample_request_preview_clean = sample_request_preview.strip().strip('"').strip("'")

json_df = next(filter(
    lambda x: x['inputs']['user_demand'] == sample_request_preview_clean, dataset_df['records']
))

json_df

StopIteration: 

In [ ]:
result

Feedback(name='TaskCompletion', source=AssessmentSource(source_type='LLM_JUDGE', source_id='openrouter:/qwen/qwen3.6-plus'), trace_id=None, run_id=None, rationale='The actual outcome fully satisfies the task by covering essential knife skills, safety guidelines, and uniform cutting techniques for vegetables, and explicitly provides the requested video IDs and timestamps in a clear, structured format.', metadata={'score': 1.0, 'threshold': 0.5, 'mlflow.scorer.framework': 'deepeval'}, span_id=None, create_time_ms=1775983746997, last_update_time_ms=1775983746997, assessment_id=None, error=None, expectation=None, feedback=FeedbackValue(value=<CategoricalRating.YES: 'yes'>, error=None), overrides=None, valid=True)

In [9]:
tool_eff_scorer = ToolCallCorrectness(model=judge_model) #type:ignore
tool_eff_result = tool_eff_scorer(trace=sample_trace)

2026/04/12 16:11:35 WARNING mlflow.genai.utils.trace_utils: Failed to extract tools from trace using LLM. Returning empty list. Error: MlflowException('Failed to invoke the judge via litellm: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-sp-af**************************cd88. You can find your API key at https://platform.openai.com/account/api-keys.')


In [10]:
tool_eff_result

Feedback(name='tool_call_correctness', source=AssessmentSource(source_type='LLM_JUDGE', source_id='openrouter:/qwen/qwen3.6-plus'), trace_id=None, run_id=None, rationale="The agent's goal is to find video segments demonstrating essential knife skills for cutting vegetables, focusing on grip, safety, and uniform cuts. The agent correctly identifies that a multi-faceted search strategy is needed, combining audio/transcript searches, visual/image searches, and knowledge graph queries. It spawns specialized workers (`transcript_keyword_search_worker`, `visual_kg_search_worker`, `deep_dive_safety_grip_worker`) with clear, detailed plans and appropriate tool selections. The individual tool calls (e.g., `search_bm25`, `get_audio_from_query_hybrid`, `get_segments_from_event_query_mmbert`, `get_images_from_qwenvl_query`, `get_related_asr_from_segment`) are highly relevant to the task. The arguments passed to these tools (such as specific culinary queries like 'knife safety', 'julienne', 'dice v

In [17]:
sample_trace

Trace(trace_id=tr-8fafb04c98bd4e967d79ff101ace7c43)

In [54]:

trace_record = get_record_based_on_trace(dataset_df['records'], sample_trace)
expectations = trace_record['expectations']


In [56]:
corr_scorer = Correctness(model=judge_model,)
corr_result = corr_scorer(trace=sample_trace, expectations=expectations,)

In [61]:
corr_result.to_dictionary()

{'assessment_name': 'correctness',
 'trace_id': '',
 'source': {'source_type': 'LLM_JUDGE',
  'source_id': 'openrouter:/qwen/qwen3.6-plus'},
 'create_time': '2026-04-12T13:52:57.002Z',
 'last_update_time': '2026-04-12T13:52:57.002Z',
 'feedback': {'value': 'no'},
 'rationale': 'The claim includes specific instructions for cutting mushrooms and leeks, but the provided document does not mention either of these vegetables; it only covers carrots, onions, herbs, bell peppers, celery, eggplant, zucchini, spring onions, and Chinese cabbage. Additionally, the video segment timestamps in the claim do not match those in the document (e.g., the claim lists 69d1f68d846a50051062bede as 00:02:16-00:03:13, while the document states 00:01:42-00:03:05, and 69d21819d7ce9f0cd06950dd as 00:00:53-00:01:28, while the document lists segments starting at 00:04:11). Due to the inclusion of unmentioned vegetables and incorrect timestamps, the claim is not supported.',
 'metadata': {'mlflow.assessment.judgeInpu

In [64]:
corr_result.feedback

FeedbackValue(value=<CategoricalRating.NO: 'no'>, error=None)